# Silver Layer — Data Cleaning & Standardisation

## Retail Banking Environment — Customer Financial Behaviour Analysis

### Context
This notebook is **Part 2 of 3** in our group project medallion pipeline.

- **Part 1 (Bronze)** — Raw mock data was generated and written to `sandbox_catalog.banking_details` as four unprocessed tables: `customers`, `accounts`, `transactions`, `loan_applications`.
- **Part 2 (Silver) ← You are here** — Each Bronze table is read, inspected for quality issues, cleaned, and written back as a Silver table ready for joining.
- **Part 3 (Gold)** — Silver tables will be joined and aggregated into analytical Gold tables for EDA.

### What This Notebook Does
For each of the four tables we will:
1. Read the Bronze table and run a null report
2. Identify and document every data-quality issue injected at the Bronze stage
3. Apply targeted cleaning steps (drop nulls, standardise casing, fix inconsistent values, cast types, filter invalid rows)
4. Write the cleaned result as a Silver Delta table

> **Note for the Gold layer team member:** All Silver tables share the same catalog and schema (`sandbox_catalog.banking_details`). Table names follow the pattern `<entity>_silver` (e.g. `customers_silver`). Foreign key columns (`customer_id`, `account_id`) are preserved as-is so joins work directly.
%md
---
## 1. Customers — Silver Cleaning

### Known Bronze Data-Quality Issues
| Column | Issue |
|---|---|
| `province` | Mixed formats — full name, abbreviation, and lowercase variants all exist (e.g. `"Alberta"`, `"AB"`, `"alberta"`) |
| `age` | No explicit nulls injected, but ages outside 18–100 should be treated as invalid |
| General | Column names are already clean; no structural issues |

### Cleaning Steps
1. Run null report on the raw Bronze table
2. Drop any rows where a required key column (`customer_id`) is null
3. Standardise `province` to the two-letter uppercase abbreviation
4. Filter out any ages outside the plausible range (18–100)
5. Write to `customers_silver`

%md
---
## 2. Accounts — Silver Cleaning

### Known Bronze Data-Quality Issues
| Column | Issue |
|---|---|
| `account_type` | Inconsistent casing and spelling — `'savings'`, `'Savings'`, `'CHECKINGS'`, `'Checkings'` |
| `status` | All lowercase already (`'active'`, `'inactive'`, `'closed'`), but should be validated against the known set |
| `balance` | Randomly generated between 1,000–1,000,000; no explicit invalid values, but zero/negative balances should be filtered |
| General | No nulls explicitly injected, but structural completeness check is still performed |

### Cleaning Steps
1. Run null report
2. Drop rows with null `account_id` or `customer_id`
3. Standardise `account_type` to lowercase (`'chequing'` / `'savings'`)
4. Validate `status` against the known set
5. Filter out non-positive balances
6. Write to `accounts_silver`

%md
---
## 2. Accounts — Silver Cleaning

### Known Bronze Data-Quality Issues
| Column | Issue |
|---|---|
| `account_type` | Inconsistent casing and spelling — `'savings'`, `'Savings'`, `'CHECKINGS'`, `'Checkings'` |
| `status` | All lowercase already (`'active'`, `'inactive'`, `'closed'`), but should be validated against the known set |
| `balance` | Randomly generated between 1,000–1,000,000; no explicit invalid values, but zero/negative balances should be filtered |
| General | No nulls explicitly injected, but structural completeness check is still performed |

### Cleaning Steps
1. Run null report
2. Drop rows with null `account_id` or `customer_id`
3. Standardise `account_type` to lowercase (`'chequing'` / `'savings'`)
4. Validate `status` against the known set
5. Filter out non-positive balances
6. Write to `accounts_silver`

%md
---
## 3. Transactions — Silver Cleaning

### Known Bronze Data-Quality Issues
| Column | Issue |
|---|---|
| `amount` | Stored as a **string** instead of a numeric type; ~3% of rows are `null`; ~5% of withdrawals have a **negative string value** (e.g. `"-123.45"`) |
| `merchant_category` | Intentionally `null` for `deposit` and `transfer` types; also randomly null for ~5% of withdrawals — these nulls for non-withdrawal types are expected, not errors |
| `transaction_date` | Stored as a string (`"YYYY-MM-DD"`) — must be cast to `DateType` |
| General | Rows with null `amount` should be dropped; negative amounts on withdrawals should be converted to their absolute value |

### Cleaning Steps
1. Run null report
2. Drop rows with null `transaction_id` or `account_id`
3. Drop rows where `amount` is null (genuinely missing data)
4. Cast `amount` from string to `DoubleType`
5. Convert any negative amounts to their absolute value (the sign is captured by `transaction_type`)
6. Cast `transaction_date` from string to `DateType`
7. Validate `transaction_type` against the known set
8. Write to `transactions_silver`


%md
---
## 4. Loan Applications — Silver Cleaning

### Known Bronze Data-Quality Issues
| Column | Issue |
|---|---|
| `amount_requested` | ~5% of rows are `null` — must be dropped |
| `interest_rate` | ~3% of rows have clearly invalid rates between 100% and 200% — must be filtered out |
| `application_date` | Stored as a string (`"YYYY-MM-DD"`) — must be cast to `DateType` |
| `loan_type` | All lowercase already; validate against the known set (`mortgage`, `auto`, `personal`) |
| `status` | All lowercase already; validate against the known set (`approved`, `denied`, `pending`) |

### Cleaning Steps
1. Run null report
2. Drop rows with null `application_id` or `customer_id`
3. Drop rows where `amount_requested` is null
4. Filter out invalid interest rates (> 30% is treated as implausible for any loan type)
5. Cast `application_date` from string to `DateType`
6. Validate `loan_type` and `status` against known sets
7. Write to `loan_applications_silver`

